# Smart Factory Sensor Data Analysis

### Made by: Santiago Villa Rodríguez

In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Spark SQL Example 1"

su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:19:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=spark://spark-master:7077 appName=Spark SQL Example 1>

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, FloatType
from datetime import datetime

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temperature", FloatType(), True)
])

df = su._spark.createDataFrame(factory_data, factory_schema)
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: float (nullable = true)



In [3]:
df.show()


+----+-------------------+-----------+
|  id|          timestamp|temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:00:00|       75.3|
|M002|2026-01-26 08:05:00|       68.7|
|M001|2026-01-26 08:10:00|       76.1|
|M003|2026-01-26 08:15:00|       72.4|
|M002|2026-01-26 08:20:00|       69.8|
|M001|2026-01-26 08:25:00|       77.5|
|M003|2026-01-26 08:30:00|       73.2|
|M002|2026-01-26 08:35:00|       70.1|
|M001|2026-01-26 08:40:00|       78.0|
|M003|2026-01-26 08:45:00|       74.6|
+----+-------------------+-----------+



### Filtering

In [4]:
from pyspark.sql.functions import col

filtered_df = df.filter(col("temperature") > 75)

filtered_df.show(truncate=False)

+----+-------------------+-----------+
|id  |timestamp          |temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:00:00|75.3       |
|M001|2026-01-26 08:10:00|76.1       |
|M001|2026-01-26 08:25:00|77.5       |
|M001|2026-01-26 08:40:00|78.0       |
+----+-------------------+-----------+



### Order By

In [5]:
df = df.orderBy(col("temperature"), ascending=False)
df.show()

+----+-------------------+-----------+
|  id|          timestamp|temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:40:00|       78.0|
|M001|2026-01-26 08:25:00|       77.5|
|M001|2026-01-26 08:10:00|       76.1|
|M001|2026-01-26 08:00:00|       75.3|
|M003|2026-01-26 08:45:00|       74.6|
|M003|2026-01-26 08:30:00|       73.2|
|M003|2026-01-26 08:15:00|       72.4|
|M002|2026-01-26 08:35:00|       70.1|
|M002|2026-01-26 08:20:00|       69.8|
|M002|2026-01-26 08:05:00|       68.7|
+----+-------------------+-----------+



### Group By

In [8]:
groupped_df = df.groupBy(col("id")).count()
groupped_df.show()

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M003|    3|
|M001|    4|
+----+-----+



### Average, Min & Max

In [12]:
from pyspark.sql.functions import avg, min, max

agg_df = df.groupBy(col("id")).agg(
    avg("temperature").alias("Avg Temp"),
    min("temperature").alias("Min Temp"),
    max("temperature").alias("Max Temp")
)

agg_df.show()

+----+-----------------+--------+--------+
|  id|         Avg Temp|Min Temp|Max Temp|
+----+-----------------+--------+--------+
|M002|69.53333282470703|    68.7|    70.1|
|M003|73.39999898274739|    72.4|    74.6|
|M001|76.72500038146973|    75.3|    78.0|
+----+-----------------+--------+--------+



In [13]:
su.stop()